
# Synthetic Patient Generation

## Research Project

**Research question:** How does the performance of HPO semantic similarity methods differ between synthetic patients and clinical cases from the literature?

### Objectives
- Generate synthetic patients from disease-phenotype profiles.
- Simulate incomplete phenotype information.
- Introduce noise into patient phenotype profiles.
- Validate the generated synthetic patients.


In [30]:
sample_disease = next(iter(disease_profiles))

print("Disease ID:", sample_disease)
print("Phenotype terms:", disease_profiles[sample_disease])
print("Number of terms:", len(disease_profiles[sample_disease]))

Disease ID: DECIPHER:1
Phenotype terms: {'HP:0001250', 'HP:0001518', 'HP:0000252', 'HP:0001249', 'HP:0001252'}
Number of terms: 5


In [31]:
profile_sizes = [
    len(profile)
    for profile in disease_profiles.values()
]

print("Number of diseases:", len(profile_sizes))
print("Minimum phenotype terms:", min(profile_sizes))
print("Maximum phenotype terms:", max(profile_sizes))
print("Average phenotype terms:", sum(profile_sizes) / len(profile_sizes))

Number of diseases: 12866
Minimum phenotype terms: 1
Maximum phenotype terms: 230
Average phenotype terms: 20.79737292087673


In [32]:
from collections import Counter

profile_size_distribution = Counter(
    len(profile)
    for profile in disease_profiles.values()
)

for size in range(1, 11):
    print(
        f"Diseases with {size} phenotype terms:",
        profile_size_distribution.get(size, 0)
    )

Diseases with 1 phenotype terms: 644
Diseases with 2 phenotype terms: 571
Diseases with 3 phenotype terms: 539
Diseases with 4 phenotype terms: 501
Diseases with 5 phenotype terms: 494
Diseases with 6 phenotype terms: 471
Diseases with 7 phenotype terms: 419
Diseases with 8 phenotype terms: 391
Diseases with 9 phenotype terms: 369
Diseases with 10 phenotype terms: 367


In [1]:

from pathlib import Path
import sys
import random

import pandas as pd
import obonet
import networkx as nx

# Define project paths
PROJECT_ROOT = Path.cwd().parent
HPO_DIR = PROJECT_ROOT / "data" / "hpo"

OBO_FILE = HPO_DIR / "hp-base.obo"
HPOA_FILE = HPO_DIR / "phenotype.hpoa"

# Make project modules accessible
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Ontology file exists:", OBO_FILE.exists())
print("Annotation file exists:", HPOA_FILE.exists())


Project root: D:\uni_ 3\ProiectCercetare\hpo-similarity-synthetic-vs-clinical
Ontology file exists: True
Annotation file exists: True


In [2]:

# Load HPO ontology
hpo_graph = obonet.read_obo(OBO_FILE)

# Load disease-phenotype annotations
annotations = pd.read_csv(
    HPOA_FILE,
    sep="\t",
    comment="#",
    dtype=str
)

# Keep positive phenotypic annotations
phenotype_annotations = annotations[
    (annotations["aspect"] == "P") &
    (annotations["qualifier"].isna())
].copy()

print("Ontology nodes:", hpo_graph.number_of_nodes())
print("Ontology edges:", hpo_graph.number_of_edges())
print("Positive phenotype annotations:", len(phenotype_annotations))


Ontology nodes: 19904
Ontology edges: 24453
Positive phenotype annotations: 268179


In [3]:

from src.disease_profiles import build_disease_profiles

disease_profiles = build_disease_profiles(
    phenotype_annotations,
    hpo_graph
)

print("Number of disease profiles:", len(disease_profiles))


Number of disease profiles: 12866


In [4]:

from src.synthetic_patients import generate_synthetic_patient


In [5]:

synthetic_patient = generate_synthetic_patient(
    disease_profiles,
    hpo_graph,
    retention_rate=0.6,
    noise_count=2,
    random_seed=42
)

print("True disease:", synthetic_patient["true_disease_id"])
print("Original phenotype terms:", synthetic_patient["original_terms"])
print("Noise terms:", synthetic_patient["noise_terms"])
print("Patient phenotype profile:", synthetic_patient["phenotype_terms"])


True disease: ORPHA:309288
Original phenotype terms: {'HP:0007957', 'HP:0000158', 'HP:0000543', 'HP:0031123', 'HP:0000545', 'HP:0001659', 'HP:0001256', 'HP:0001433', 'HP:0002120', 'HP:0001876', 'HP:0002329', 'HP:0000518', 'HP:0002312', 'HP:0000738', 'HP:0000739', 'HP:0000410', 'HP:0025406', 'HP:0002719'}
Noise terms: {'HP:0100933', 'HP:6001410'}
Patient phenotype profile: {'HP:0007957', 'HP:0000158', 'HP:0000543', 'HP:0001433', 'HP:0000545', 'HP:0001876', 'HP:0000739', 'HP:0000410', 'HP:0002719', 'HP:0031123', 'HP:0002120', 'HP:0001659', 'HP:0001256', 'HP:0100933', 'HP:0002329', 'HP:0000518', 'HP:6001410', 'HP:0002312', 'HP:0000738', 'HP:0025406'}


In [6]:

true_disease = synthetic_patient["true_disease_id"]
original_terms = synthetic_patient["original_terms"]
noise_terms = synthetic_patient["noise_terms"]
patient_terms = synthetic_patient["phenotype_terms"]

assert true_disease in disease_profiles

assert len(original_terms) >= 1
assert len(noise_terms) == 2

assert original_terms.issubset(disease_profiles[true_disease])
assert noise_terms.isdisjoint(disease_profiles[true_disease])

assert patient_terms == original_terms | noise_terms

assert all(term in hpo_graph for term in patient_terms)

print("All synthetic patient validation tests passed.")


All synthetic patient validation tests passed.


In [7]:
print("Available columns:")
print(phenotype_annotations.columns.tolist())

print("\nFrequency examples:")
print(phenotype_annotations["frequency"].value_counts(dropna=False).head(15))

print("\nSample annotations:")
display(
    phenotype_annotations[
        ["database_id", "hpo_id", "frequency"]
    ].head(15)
)

Available columns:
['database_id', 'disease_name', 'qualifier', 'hpo_id', 'reference', 'evidence', 'onset', 'frequency', 'sex', 'modifier', 'aspect', 'biocuration']

Frequency examples:
frequency
NaN           51902
HP:0040283    45420
HP:0040282    40398
HP:0040281    25753
1/1           18964
2/2            7337
HP:0040284     7026
1/2            5901
3/3            4016
1/3            3966
1/4            3205
4/4            2748
1/5            2212
2/3            1860
1/6            1722
Name: count, dtype: int64

Sample annotations:


,database_id,hpo_id,frequency
0,OMIM:619340,HP:0011097,1/2
1,OMIM:619340,HP:0002187,1/1
2,OMIM:619340,HP:0001518,1/2
3,OMIM:619340,HP:0032792,1/2
4,OMIM:619340,HP:0011451,1/2
5,OMIM:619340,HP:0010851,2/2
6,OMIM:619340,HP:0001789,1/2
7,OMIM:619340,HP:0200134,2/2
10,OMIM:619340,HP:0002643,2/2
11,OMIM:609153,HP:0002378,NaN


In [8]:
frequency_terms = [
    "HP:0040281",
    "HP:0040282",
    "HP:0040283",
    "HP:0040284"
]

for term in frequency_terms:
    print(term, "->", hpo_graph.nodes[term].get("name"))

HP:0040281 -> Very frequent
HP:0040282 -> Frequent
HP:0040283 -> Occasional
HP:0040284 -> Very rare


In [9]:
for term in frequency_terms:
    print(f"\n{term}")
    print(hpo_graph.nodes[term])


HP:0040281
{'name': 'Very frequent', 'def': '"Present in 80% to 99% of the cases." []', 'synonym': ['"Very frequent (99-80%)" EXACT []'], 'xref': ['ORPHA:453311'], 'is_a': ['HP:0040279'], 'property_value': ['dcterms:creator https://orcid.org/0000-0002-5316-1399', 'dcterms:date "2016-12-01T11:44:56Z" xsd:dateTime']}

HP:0040282
{'name': 'Frequent', 'def': '"Present in 30% to 79% of the cases." []', 'synonym': ['"Frequent (79-30%)" EXACT []'], 'xref': ['ORPHA:453312'], 'is_a': ['HP:0040279'], 'property_value': ['dcterms:creator https://orcid.org/0000-0002-5316-1399', 'dcterms:date "2016-12-01T11:44:56Z" xsd:dateTime']}

HP:0040283
{'name': 'Occasional', 'def': '"Present in 5% to 29% of the cases." []', 'synonym': ['"Occasional (29-5%)" EXACT []'], 'xref': ['ORPHA:453313'], 'is_a': ['HP:0040279'], 'property_value': ['dcterms:creator https://orcid.org/0000-0002-5316-1399', 'dcterms:date "2016-12-01T11:44:56Z" xsd:dateTime']}

HP:0040284
{'name': 'Very rare', 'def': '"Present in 1% to 4% o

In [10]:
# Check duplicate disease-phenotype pairs
duplicate_pairs = phenotype_annotations.duplicated(
    subset=["database_id", "hpo_id"],
    keep=False
)

print("Total annotations:", len(phenotype_annotations))
print("Annotations in duplicate pairs:", duplicate_pairs.sum())

display(
    phenotype_annotations.loc[
        duplicate_pairs,
        ["database_id", "hpo_id", "frequency"]
    ].head(20)
)

Total annotations: 268179
Annotations in duplicate pairs: 1046


,database_id,hpo_id,frequency
124,OMIM:612567,HP:0025084,2/2
125,OMIM:612567,HP:0025084,1/1
152,OMIM:614116,HP:0000726,18/18
153,OMIM:614116,HP:0000726,NaN
1233,OMIM:122600,HP:0002650,3/3
1234,OMIM:122600,HP:0002650,23/23
1789,OMIM:616271,HP:0001252,8/14
1790,OMIM:616271,HP:0001252,3/5
3216,OMIM:619817,HP:0007589,1/1
3217,OMIM:619817,HP:0007589,1/1


In [11]:
duplicate_details = phenotype_annotations.loc[
    duplicate_pairs,
    [
        "database_id",
        "hpo_id",
        "frequency",
        "reference",
        "evidence",
        "biocuration"
    ]
]

display(duplicate_details.head(30))

,database_id,hpo_id,frequency,reference,evidence,biocuration
124,OMIM:612567,HP:0025084,2/2,PMID:19890111,PCS,HPO:skoehler[2018-10-08];HPO:probinson[2020-11...
125,OMIM:612567,HP:0025084,1/1,PMID:21519361,PCS,HPO:probinson[2022-02-27]
152,OMIM:614116,HP:0000726,18/18,PMID:21532572,PCS,HPO:skoehler[2012-11-20]
153,OMIM:614116,HP:0000726,NaN,OMIM:614116,TAS,HPO:skoehler[2012-11-20]
1233,OMIM:122600,HP:0002650,3/3,PMID:23335591,PCS,HPO:probinson[2015-02-21];HPO:probinson[2022-0...
1234,OMIM:122600,HP:0002650,23/23,PMID:25564734,PCS,HPO:probinson[2022-08-03]
1789,OMIM:616271,HP:0001252,8/14,PMID:25597510,PCS,HPO:skoehler[2015-05-31];HPO:probinson[2022-06...
1790,OMIM:616271,HP:0001252,3/5,PMID:25597511,PCS,HPO:probinson[2022-06-04]
3216,OMIM:619817,HP:0007589,1/1,PMID:9185503,PCS,HPO:probinson[2022-05-12]
3217,OMIM:619817,HP:0007589,1/1,PMID:14675179,PCS,HPO:probinson[2022-05-12]


In [12]:
# Count annotations per disease-phenotype pair
pair_counts = (
    phenotype_annotations
    .groupby(["database_id", "hpo_id"])
    .size()
)

print("Total disease-phenotype pairs:", len(pair_counts))
print("Pairs with multiple annotations:", (pair_counts > 1).sum())
print("Maximum annotations for one pair:", pair_counts.max())

print("\nDistribution of annotations per pair:")
print(pair_counts.value_counts().sort_index())

Total disease-phenotype pairs: 267579
Pairs with multiple annotations: 446
Maximum annotations for one pair: 16

Distribution of annotations per pair:
1     267133
2        355
3         65
4         13
5          8
6          1
7          2
13         1
16         1
Name: count, dtype: int64


In [13]:

from src.phenotype_frequency import frequency_to_probability

test_frequencies = [
    "1/2",
    "2/2",
    "3/5",
    "HP:0040281",
    "HP:0040282",
    "HP:0040283",
    "HP:0040284",
    None
]

for frequency in test_frequencies:
    print(f"{frequency} -> {frequency_to_probability(frequency)}")


1/2 -> 0.5
2/2 -> 1.0
3/5 -> 0.6
HP:0040281 -> 0.895
HP:0040282 -> 0.545
HP:0040283 -> 0.16999999999999998
HP:0040284 -> 0.025
None -> None


In [15]:
import importlib
import src.phenotype_frequency

importlib.reload(src.phenotype_frequency)

from src.phenotype_frequency import (
    frequency_to_probability,
    aggregate_phenotype_frequencies
)

test_frequencies = [
    "1/2",
    "3/5",
    "25%",
    "80%",
    "100%",
    "HP:0040280",
    "HP:0040281",
    "HP:0040282",
    "HP:0040283",
    "HP:0040284",
    None
]

for value in test_frequencies:
    print(value, "->", frequency_to_probability(value))

1/2 -> 0.5
3/5 -> 0.6
25% -> 0.25
80% -> 0.8
100% -> 1.0
HP:0040280 -> 1.0
HP:0040281 -> 0.895
HP:0040282 -> 0.545
HP:0040283 -> 0.16999999999999998
HP:0040284 -> 0.025
None -> None


In [16]:

from src.phenotype_frequency import aggregate_phenotype_frequencies

aggregated_frequencies = aggregate_phenotype_frequencies(
    phenotype_annotations
)

print("Original annotations:", len(phenotype_annotations))
print("Aggregated disease-phenotype pairs:", len(aggregated_frequencies))

display(aggregated_frequencies.head())


Original annotations: 268179
Aggregated disease-phenotype pairs: 267579


,database_id,hpo_id,frequency,annotation_count,unknown_frequency_count
0,DECIPHER:1,HP:0000252,NaN,1,1
1,DECIPHER:1,HP:0001249,NaN,1,1
2,DECIPHER:1,HP:0001250,NaN,1,1
3,DECIPHER:1,HP:0001252,NaN,1,1
4,DECIPHER:1,HP:0001518,NaN,1,1


In [17]:
print("Known frequencies:",
      aggregated_frequencies["frequency"].notna().sum())

print("Unknown frequencies:",
      aggregated_frequencies["frequency"].isna().sum())

print("\nFrequency statistics:")
display(aggregated_frequencies["frequency"].describe())

Known frequencies: 215736
Unknown frequencies: 51843

Frequency statistics:


count    215736.000000
mean          0.530141
std           0.341363
min           0.000000
25%           0.170000
50%           0.545000
75%           0.895000
max           1.000000
Name: frequency, dtype: float64

In [18]:
aggregated_frequencies["frequency"].describe()

count    215736.000000
mean          0.530141
std           0.341363
min           0.000000
25%           0.170000
50%           0.545000
75%           0.895000
max           1.000000
Name: frequency, dtype: float64

In [19]:
print(
    "Pairs with frequency 0:",
    (aggregated_frequencies["frequency"] == 0).sum()
)

print(
    "Pairs with frequency 1:",
    (aggregated_frequencies["frequency"] == 1).sum()
)

Pairs with frequency 0: 3018
Pairs with frequency 1: 41210


In [20]:
disease_frequency_stats = (
    aggregated_frequencies
    .assign(
        is_zero=lambda df: df["frequency"] == 0,
        is_positive=lambda df: df["frequency"] > 0,
        is_unknown=lambda df: df["frequency"].isna()
    )
    .groupby("database_id")
    .agg(
        total_phenotypes=("hpo_id", "count"),
        known_positive=("is_positive", "sum"),
        zero_frequency=("is_zero", "sum"),
        unknown_frequency=("is_unknown", "sum")
    )
    .reset_index()
)

display(disease_frequency_stats.head(10))

,database_id,total_phenotypes,known_positive,zero_frequency,unknown_frequency
0,DECIPHER:1,5,0,0,5
1,DECIPHER:14,5,0,0,5
2,DECIPHER:15,6,0,0,6
3,DECIPHER:16,5,0,0,5
4,DECIPHER:17,5,0,0,5
5,DECIPHER:18,6,0,0,6
6,DECIPHER:19,4,0,0,4
7,DECIPHER:2,3,0,0,3
8,DECIPHER:20,5,0,0,5
9,DECIPHER:21,5,0,0,5


In [21]:
print("Number of diseases:", len(disease_frequency_stats))

print("\nTotal phenotypes per disease:")
display(disease_frequency_stats["total_phenotypes"].describe())

print("\nKnown positive phenotypes per disease:")
display(disease_frequency_stats["known_positive"].describe())

print("\nZero-frequency phenotypes per disease:")
display(disease_frequency_stats["zero_frequency"].describe())

print("\nUnknown-frequency phenotypes per disease:")
display(disease_frequency_stats["unknown_frequency"].describe())

Number of diseases: 12866

Total phenotypes per disease:


count    12866.000000
mean        20.797373
std         20.089846
min          1.000000
25%          6.000000
50%         15.000000
75%         28.000000
max        230.000000
Name: total_phenotypes, dtype: float64


Known positive phenotypes per disease:


count    12866.000000
mean        16.533344
std         19.452694
min          0.000000
25%          2.000000
50%         11.000000
75%         24.000000
max        226.000000
Name: known_positive, dtype: float64


Zero-frequency phenotypes per disease:


count    12866.000000
mean         0.234572
std          1.152906
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         49.000000
Name: zero_frequency, dtype: float64


Unknown-frequency phenotypes per disease:


count    12866.000000
mean         4.029457
std          7.813905
min          0.000000
25%          0.000000
50%          0.000000
75%          5.000000
max         80.000000
Name: unknown_frequency, dtype: float64

In [22]:
print(disease_frequency_stats[
    ["total_phenotypes", "known_positive",
     "zero_frequency", "unknown_frequency"]
].describe().to_string())

       total_phenotypes  known_positive  zero_frequency  unknown_frequency
count      12866.000000    12866.000000    12866.000000       12866.000000
mean          20.797373       16.533344        0.234572           4.029457
std           20.089846       19.452694        1.152906           7.813905
min            1.000000        0.000000        0.000000           0.000000
25%            6.000000        2.000000        0.000000           0.000000
50%           15.000000       11.000000        0.000000           0.000000
75%           28.000000       24.000000        0.000000           5.000000
max          230.000000      226.000000       49.000000          80.000000


In [23]:
diseases_without_known_frequency = disease_frequency_stats[
    disease_frequency_stats["known_positive"] == 0
]

print(
    "Diseases without known positive frequencies:",
    len(diseases_without_known_frequency)
)

print(
    "\nPercentage:",
    round(
        len(diseases_without_known_frequency)
        / len(disease_frequency_stats) * 100,
        2
    ),
    "%"
)

display(diseases_without_known_frequency.head(10))

Diseases without known positive frequencies: 2548

Percentage: 19.8 %


,database_id,total_phenotypes,known_positive,zero_frequency,unknown_frequency
0,DECIPHER:1,5,0,0,5
1,DECIPHER:14,5,0,0,5
2,DECIPHER:15,6,0,0,6
3,DECIPHER:16,5,0,0,5
4,DECIPHER:17,5,0,0,5
5,DECIPHER:18,6,0,0,6
6,DECIPHER:19,4,0,0,4
7,DECIPHER:2,3,0,0,3
8,DECIPHER:20,5,0,0,5
9,DECIPHER:21,5,0,0,5


In [24]:
import importlib
import src.phenotype_frequency

importlib.reload(src.phenotype_frequency)

from src.phenotype_frequency import (
    build_disease_frequency_profiles
)

disease_frequency_profiles = build_disease_frequency_profiles(
    aggregated_frequencies
)

In [25]:
print("Number of diseases:", len(disease_frequency_profiles))

sample_disease = "DECIPHER:1"

print("\nDisease:", sample_disease)
print("Phenotype frequencies:")
print(disease_frequency_profiles[sample_disease])

print(
    "\nNumber of phenotypes:",
    len(disease_frequency_profiles[sample_disease])
)

Number of diseases: 12866

Disease: DECIPHER:1
Phenotype frequencies:
{'HP:0000252': None, 'HP:0001249': None, 'HP:0001250': None, 'HP:0001252': None, 'HP:0001518': None}

Number of phenotypes: 5


In [26]:
sample_mixed_disease = next(
    disease_id
    for disease_id, frequencies in disease_frequency_profiles.items()
    if any(value is not None for value in frequencies.values())
    and any(value is None for value in frequencies.values())
)

print("Disease:", sample_mixed_disease)
print("Phenotype frequencies:")
print(disease_frequency_profiles[sample_mixed_disease])

Disease: OMIM:100100
Phenotype frequencies:
{'HP:0000028': None, 'HP:0000072': None, 'HP:0000126': None, 'HP:0000217': 1.0, 'HP:0000767': None, 'HP:0000768': None, 'HP:0001374': None, 'HP:0001562': None, 'HP:0001643': None, 'HP:0001762': None, 'HP:0002023': None, 'HP:0004392': 0.4, 'HP:0005199': None, 'HP:0010957': 1.0, 'HP:0030211': 1.0}


In [27]:
expected_phenotypes_per_disease = (
    aggregated_frequencies
    .dropna(subset=["frequency"])
    .groupby("database_id")["frequency"]
    .sum()
)

mean_expected_phenotypes = expected_phenotypes_per_disease.mean()

print(
    "Diseases with known frequencies:",
    len(expected_phenotypes_per_disease)
)

print(
    "Mean expected selected phenotypes:",
    mean_expected_phenotypes
)

Diseases with known frequencies: 10338
Mean expected selected phenotypes: 11.06311101199003


In [28]:
# Diseases where every phenotype frequency is unknown
fully_unknown_diseases = disease_frequency_stats[
    (disease_frequency_stats["unknown_frequency"] ==
     disease_frequency_stats["total_phenotypes"])
]

# Diseases with known frequencies, but no positive frequency
no_positive_frequency_diseases = disease_frequency_stats[
    (disease_frequency_stats["known_positive"] == 0) &
    (disease_frequency_stats["unknown_frequency"] <
     disease_frequency_stats["total_phenotypes"])
]

print("Diseases with all frequencies unknown:",
      len(fully_unknown_diseases))

print("Diseases with known frequencies but no positive values:",
      len(no_positive_frequency_diseases))

Diseases with all frequencies unknown: 2528
Diseases with known frequencies but no positive values: 20


In [29]:
eligible_diseases = disease_frequency_stats[
    disease_frequency_stats["known_positive"] > 0
]["database_id"]

mean_expected_phenotypes = (
    aggregated_frequencies[
        aggregated_frequencies["database_id"].isin(eligible_diseases)
    ]
    .groupby("database_id")["frequency"]
    .sum()
    .mean()
)

print("Eligible diseases:", len(eligible_diseases))
print("Mean expected selected phenotypes:", mean_expected_phenotypes)

Eligible diseases: 10318
Mean expected selected phenotypes: 11.084555305480997
